# Motion-Guided Self-Supervised Learning for Cardiac Cine MRI
## Notebook 04: Self-Supervised Temporal Pretraining

This notebook provides a complete demonstration and interactive inspection of the **Self-Supervised Temporal Representation-Learning Pipeline**.

### Pretraining Objectives:
1. **Masked Image Reconstruction**: Divides cine slices into non-overlapping patches (e.g. 16x16), randomly masks 50% of them, and trains the shared encoder + reconstruction decoder to predict missing cardiac anatomy.
2. **Adjacent-Frame Temporal Feature Consistency**: Exploits natural cardiac cycle continuity by pairing temporally adjacent frames $(t, t+1)$ from the same patient and slice position, penalizing divergence in the projected embedding space.

> **COMPUTE CONSTRAINT NOTE**:
> Per project guidelines, full multi-epoch SSL pretraining is executed on a dedicated GPU training server (`TRAINING MACHINE ONLY`).
> This notebook runs a lightweight CPU demonstration without executing expensive training loops.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is in sys.path
project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import yaml
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from src.ssl import SSLModel, PatchMasker, ReconstructionDecoder, compute_ssl_loss, count_parameters
from src.dataset import ACDCTemporalDataset

print(f"Project root: {project_root}")
print(f"PyTorch version: {torch.__version__}")

### 1. Load Pretraining Configuration
We inspect the dedicated `configs/ssl.yaml` configuration file.

In [ ]:
config_path = project_root / "configs" / "ssl.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("SSL Configuration:")
print(f"  Encoder channels:   {config['model']['encoder_channels']}")
print(f"  Projection dim:     {config['model']['proj_dim']}")
print(f"  Mask patch size:    {config['model']['mask_patch_size']} x {config['model']['mask_patch_size']}")
print(f"  Mask ratio:         {config['model']['mask_ratio'] * 100:.0f}%")
print(f"  Recon weight:       {config['loss']['recon_weight']}")
print(f"  Temporal weight:    {config['loss']['temporal_weight']}")
print(f"  Checkpoints target: {config['logging']['checkpoint_dir']}")

### 2. Temporal Dataset: Strict Adjacent Frame Pairing $(t, t+1)$
The `ACDCTemporalDataset` indexes all cine slices across time and couples frames $(t, t+1)$ from the **exact same patient and slice position**.
Frames are never paired across different patients, ensuring clean cardiac contraction/relaxation kinetics.

In [ ]:
train_split = project_root / config['data']['train_split']
processed_dir = project_root / config['data']['processed_dir']

dataset = ACDCTemporalDataset(
    processed_dir=str(processed_dir),
    split_file=str(train_split),
)

print(f"Total training adjacent cine pairs: {len(dataset):,}")

# Inspect a random pair
sample = dataset[100]
print(f"Sample Patient:   {sample['patient_id']}")
print(f"Slice Index:      {sample['slice_idx']}")
print(f"Frame t:          {sample['frame_idx_t']}")
print(f"Frame t+1:        {sample['frame_idx_t1']}")
print(f"Tensor shape:     {sample['frame_t'].shape}")

### 3. Visualizing Adjacent Cardiac Cine Frames and Motion Difference
Below we visualize frame $t$, frame $t+1$, and their absolute intensity difference $|x_{t+1} - x_t|$ which highlights myocardial displacement.

In [ ]:
img_t = sample['frame_t'].squeeze().numpy()
img_t1 = sample['frame_t1'].squeeze().numpy()
diff = np.abs(img_t1 - img_t)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_t, cmap='gray')
axes[0].set_title(f"Frame t (Index {sample['frame_idx_t']})")
axes[0].axis('off')

axes[1].imshow(img_t1, cmap='gray')
axes[1].set_title(f"Frame t+1 (Index {sample['frame_idx_t1']})")
axes[1].axis('off')

im = axes[2].imshow(diff, cmap='inferno')
axes[2].set_title("Absolute Difference |t+1 - t| (Motion)")
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

### 4. Visualizing Patch Masking
The `PatchMasker` divides the $256 \times 256$ image into non-overlapping $16 \times 16$ patches (a $16 \times 16$ grid $= 256$ patches in total). Exactly 50% ($128$ patches) are masked out with zero intensity.

In [ ]:
masker = PatchMasker(patch_size=16, mask_ratio=0.50)
x = sample['frame_t'].unsqueeze(0)  # (1, 1, 256, 256)
masked_x, mask = masker(x)

orig_np = x.squeeze().numpy()
mask_np = mask.squeeze().numpy()
masked_np = masked_x.squeeze().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(orig_np, cmap='gray')
axes[0].set_title("Original Cine Slice (t)")
axes[0].axis('off')

axes[1].imshow(mask_np, cmap='Reds', alpha=0.8)
axes[1].set_title(f"Binary Patch Mask (Ratio: {mask_np.mean()*100:.1f}%)")
axes[1].axis('off')

axes[2].imshow(masked_np, cmap='gray')
axes[2].set_title("Masked Input to Encoder")
axes[2].axis('off')
plt.tight_layout()
plt.show()

### 5. SSL Model Architecture & Forward Pass
We instantiate the `SSLModel` on CPU and verify parameter counts.

In [ ]:
model = SSLModel(
    in_channels=1,
    encoder_channels=[32, 64, 128, 256],
    proj_dim=128,
    mask_patch_size=16,
    mask_ratio=0.50,
    dropout=0.1,
    use_residual=True,
)
model.eval()

tot_p = count_parameters(model)
enc_p = count_parameters(model.encoder)
dec_p = count_parameters(model.recon_decoder)
proj_p = count_parameters(model.projection)

print(f"Total Trainable Parameters:    {tot_p:,}")
print(f"Shared Encoder (transferable): {enc_p:,} ({enc_p/tot_p*100:.1f}%)")
print(f"Reconstruction Decoder:        {dec_p:,} ({dec_p/tot_p*100:.1f}%)")
print(f"Projection Head (temporal):    {proj_p:,} ({proj_p/tot_p*100:.1f}%)")

### 6. Masked Reconstruction Visualization
We execute a single forward pass and visualize the original frame, the masked frame, and the reconstructed frame (from initial un-trained weights, showing the operational forward flow).

In [ ]:
x_t = sample['frame_t'].unsqueeze(0)
x_t1 = sample['frame_t1'].unsqueeze(0)

with torch.no_grad():
    results = model(x_t, x_t1)

recon_np = results['reconstructed'].squeeze().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(orig_np, cmap='gray')
axes[0].set_title("Original Slice (Target)")
axes[0].axis('off')

axes[1].imshow(masked_np, cmap='gray')
axes[1].set_title("Masked Slice (Input)")
axes[1].axis('off')

axes[2].imshow(recon_np, cmap='gray')
axes[2].set_title("Decoder Output (Reconstruction)")
axes[2].axis('off')
plt.tight_layout()
plt.show()

### 7. Temporal Embedding Consistency & Loss Formulation
The projection head projects bottleneck representations to unit sphere vectors $z_t, z_{t+1} \in \mathbb{R}^{128}$:
$$\mathcal{L}_{\text{temporal}} = \| z_t - z_{t+1} \|_2^2 = 2 - 2 \cos(z_t, z_{t+1})$$
$$\mathcal{L}_{\text{ssl}} = \lambda_{\text{recon}} \mathcal{L}_{\text{recon}} + \lambda_{\text{temporal}} \mathcal{L}_{\text{temporal}}$$

In [ ]:
z_t = results['proj_t']
z_t1 = results['proj_t1']

norm_t = torch.norm(z_t, dim=-1).item()
norm_t1 = torch.norm(z_t1, dim=-1).item()
cos_sim = F.cosine_similarity(z_t, z_t1, dim=-1).item()

print(f"Embedding shape:          {z_t.shape}")
print(f"L2 norm z_t:              {norm_t:.4f} (unit sphere)")
print(f"L2 norm z_t+1:            {norm_t1:.4f} (unit sphere)")
print(f"Cosine similarity (t,t+1): {cos_sim:.4f}")

recon_loss, temp_loss, total_loss = compute_ssl_loss(
    results,
    recon_weight=1.0,
    temporal_weight=0.1,
    recon_loss_type="l1"
)

print(f"\nCalculated SSL Losses:")
print(f"  Reconstruction Loss (L1): {recon_loss.item():.4f}")
print(f"  Temporal Consistency:     {temp_loss.item():.4f}")
print(f"  Total Weighted SSL Loss:  {total_loss.item():.4f}")

### 8. Full Training on GPU System (`TRAINING MACHINE ONLY`)

To execute full SSL pretraining (100 epochs, AdamW, cosine annealing, mixed precision) on the dedicated training server, run:

```bash
# ========================================================
# TRAINING MACHINE ONLY (DO NOT RUN ON DEV SYSTEM)
# ========================================================
python src/ssl.py --config configs/ssl.yaml --device cuda
```

Checkpoints will automatically be saved to:
- `checkpoints/ssl/ssl_checkpoint_epoch{N}.pth` (periodic full state)
- `checkpoints/ssl/ssl_best.pth` (lowest SSL loss full state)
- `checkpoints/ssl/ssl_encoder_best.pth` (standalone encoder weights for downstream segmentation)